# Monoclinal Geological Model Notebook

## Overview
This notebook creates a 3D monoclinal geological model using Gempy. It generates parallel inclined layers based on parameters you can change.

## User Inputs
- **Dip Angle**: Set `angle_deg` for layer inclination.
- **Surfaces**: Modify `surface_names` and `surface_colors` lists to add/remove layers.
- **Thickness**: Adjust `layer_thickness` for layer spacing. 
- **base_z_start**: Height of the first layer' base
- **Extent & Resolution**: Customize model `extent` and `resolution` as needed for fitting more or less layers in the model.

## How to Run
1. Make sure every dependencies are installed correctly, including pytorch, as gempy relies on it for model computation.

    You should be using the GP-env kernel if you are using the FabLab computer as it is already set up for gempy.

    Otherwise check the first cell for the needed imports.

    Command to run in your environment (I'm using conda with python 3.14.3): `pip install numpy matplotlib pandas pyvista gempy gempy_viewer` 

2. Update the input variables in the second cell.
3. Execute all cells sequentially.
4. For near-vertical angles (≥85°), the model switches to vertical planes.

## Outputs
- 3D plots showing data points, orientations, and computed surfaces.

In [1]:
import os

os.environ["DEFAULT_BACKEND"] = "PYTORCH"

#Even though pytorch is not imported, you need to install it, refer to : https://pytorch.org/get-started/locally/
#In most cases, `pip3 install torch torchvision` will work with newer computers. 

import gempy as gp
import gempy_viewer as gpv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyvista as pv
#

# print("gempy version: ", gp.__version__)
# print("gempy_viewer version: ", gpv.__version__)

Setting Backend To: AvailableBackends.PYTORCH


In [2]:
monoclinal_model = gp.create_geomodel(
    project_name="MonoclinalModel",
    extent=[0, 1000, 0, 1000, 0, 1000],
    # refinement = 4,           #Chose between refinement by Octree levels or resolution
    resolution = [50, 50, 50], 
    structural_frame = gp.data.StructuralFrame.initialize_default_structure()
) 

# ------- USER INPUT SECTION -------

#User input: dip angle in degrees for parallel inclined layers
angle_deg = 25 #Must be [0°,90°]. If > 90°, returns vertical layers
angle_rad = np.deg2rad(angle_deg)

#You can add or remove surfaces by adding a name and a color, you might need to ajust the thickness and model extent when adding layers though!
#The model should not break if the extent is modified, maybe if the extent becomes huge it will. I tested up untill [0, 100000,...] and it worked.
surface_names = ["Tithonic", "Berriasian", "Valanginian", "Kimmeridgian"]
surface_colors = ["#00FFFF", "#006400", "#008000", "#00FF00"]

#Layer thickness: specify perpendicular thickness for each layer
#Make sure the number of thicknesses matches the number of surface names!
layer_thicknesses = [350.0, 200.0, 150.0, 150.0]  #Perpendicular thickness for each layer
base_z_start = 100.0


In [3]:
#Handle near-vertical angles (>= 85°), it breaks and the model and becomes a line (1D) because tan(0) and cos(0) returns 0 !
if angle_deg >= 85.0:
    #Layers are vertical planes perpendicular to x-axis
    #Calculate cumulative x-positions based on layer thickness
    x_positions = [base_z_start + sum(layer_thicknesses[:i]) for i in range(len(surface_names))]
    
    #Ensure x-positions stay within domain [100, 900], increase this bound if you add more layers #TODO Make this a user input ?
    if max(x_positions) > 900:
        # Scale down to fit
        scale_factor = 900.0 / max(x_positions)
        x_positions = [x * scale_factor for x in x_positions]
    
    #Create grid: each layer is a vertical plane with y-z points
    x_grid_vals = np.array(x_positions)
    y_grid = np.linspace(0.0, 1000.0, 8)
    z_grid = np.linspace(100.0, 900.0, 8)
    y_mesh, z_mesh = np.meshgrid(y_grid, z_grid)
    
    #Empty lists for adding points
    x = []
    y = []
    z = []
    element_names_list = []
    
    #Appends layer_i name, color, x_pos then iterates, else, dummy layer
    for idx, (name, color, x_pos) in enumerate(zip(surface_names, surface_colors, x_positions)):
        if idx == 0:
            monoclinal_model.structural_frame.structural_elements[0].name = name
            monoclinal_model.structural_frame.structural_elements[0].color = color
        else:
            element = gp.data.StructuralElement(
                name=name,
                color=color,
                surface_points=gp.data.SurfacePointsTable.initialize_empty(),
                orientations=gp.data.OrientationsTable.initialize_empty()
            )
            monoclinal_model.structural_frame.structural_groups[0].append_element(element)
        
        #Add all y-z points for this vertical plane
        x.extend([x_pos] * (8 * 8))
        y.extend(y_mesh.flatten().tolist())
        z.extend(z_mesh.flatten().tolist())
        element_names_list.extend([name] * (8 * 8))
    
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)
    element_names = element_names_list
    
    gp.add_surface_points(
        geo_model=monoclinal_model,
        x=x,
        y=y,
        z=z,
        elements_names=element_names
    )
    
    #Orientation for vertical layers: normal vector is [1, 0, 0] (pointing in x) #TODO #DONE Simpify error handling angles >= 85° by creating array of vertical points ?
    pole_vector = np.array([1.0, 0.0, 0.0])
    number_of_surfaces = len(surface_names)
    
    #Place orientation points on each vertical plane
    orientation_x = np.array(x_positions * 3)
    orientation_y = np.array([250.0, 500.0, 750.0] * number_of_surfaces)
    orientation_z = np.array([500.0, 500.0, 500.0] * number_of_surfaces)
    orientation_names = np.repeat(surface_names, 3)
    
    gp.add_orientations(
        geo_model=monoclinal_model,
        x=orientation_x,
        y=orientation_y,
        z=orientation_z,
        elements_names=orientation_names,
        pole_vector=np.vstack([pole_vector] * len(orientation_x))
    )

else:
    #Non-vertical layers: calculate base z positions using cumulative layer thicknesses
    vertical_spacings = [thickness / np.cos(angle_rad) for thickness in layer_thicknesses]
    base_zs = [base_z_start]
    for spacing in vertical_spacings[:-1]:
        base_zs.append(base_zs[-1] + spacing)
    
    #Calculate maximum horizontal extent to keep points within z bounds [0, 1000]
    max_z_rise = 1000.0 - max(base_zs) - 50.0
    max_horizontal_extent = max_z_rise / np.tan(angle_rad)
    
    #Same for X; Y is then constrained because X ans Z are also bound
    x_margin = 50.0
    max_x_extent = 1000.0 - 2 * x_margin
    horizontal_extent = min(max_x_extent, max_horizontal_extent)
    
    #Creates grid of points to make sure the layers don't bend because of gempy global interpolation, maybe overkill? It works, won't touch it more 
    x_center = 500.0
    x0 = x_center - horizontal_extent / 2.0
    x1 = x_center + horizontal_extent / 2.0
    
    x_grid = np.linspace(x0, x1, 8)
    y_grid = np.linspace(0.0, 1000.0, 8)
    x_mesh, y_mesh = np.meshgrid(x_grid, y_grid)
    x = x_mesh.flatten() #Thanks copilot for .flatten()
    y = y_mesh.flatten()
    slope = np.tan(angle_rad)
    
    #Same as for the vertical layers but with base_zs
    for idx, (name, color, base_z) in enumerate(zip(surface_names, surface_colors, base_zs)):
        if idx == 0:
            monoclinal_model.structural_frame.structural_elements[0].name = name
            monoclinal_model.structural_frame.structural_elements[0].color = color
        else:
            element = gp.data.StructuralElement(
                name=name,
                color=color,
                surface_points=gp.data.SurfacePointsTable.initialize_empty(),
                orientations=gp.data.OrientationsTable.initialize_empty()
            )
            monoclinal_model.structural_frame.structural_groups[0].append_element(element)

        #Iterates points for different heights and appends points in model
        z = base_z + (x - x0) * slope
        element_names = [name] * len(x)
        gp.add_surface_points(
            geo_model=monoclinal_model,
            x=x,
            y=y,
            z=z,
            elements_names=element_names
        )
    #Layer orientation with pole vector, thanks linear algebra for rotation matrix
    pole_vector = np.array([-np.sin(angle_rad), 0.0, np.cos(angle_rad)])

    number_of_surfaces = len(surface_names)
    
    #Iterates placement of vector poles for each layers
    #Very rigid way to do it, but there are enough vectors so that the surfaces don't bend, no need to append vectors in another direction (for now?)
    orientation_x = np.array([300.0, 500.0, 700.0] * number_of_surfaces)
    orientation_y = np.array([500.0, 500.0, 500.0] * number_of_surfaces)
    orientation_z = np.repeat(np.array(base_zs), 3) + (orientation_x - x0) * slope
    orientation_names = np.repeat(surface_names, 3)
    
    #Appends
    gp.add_orientations(
        geo_model=monoclinal_model,
        x=orientation_x,
        y=orientation_y,
        z=orientation_z,
        elements_names=orientation_names,
        pole_vector=np.vstack([pole_vector] * len(orientation_x))
    )

#Add to groupe
monoclinal_model.structural_frame.structural_groups[0].name = "Stratigraphic. Units"


# monoclinal_model.structural_frame.structural_elements[0]

monoclinal_model.structural_frame #uncomment to see the structural frame

StructuralFrame(
	structural_groups=[
StructuralGroup(
	name=Stratigraphic. Units,
	structural_relation=StackRelationType.ERODE,
	elements=[
Element(
	name=Tithonic,
	color=#00FFFF,
	is_active=True
),
Element(
	name=Berriasian,
	color=#006400,
	is_active=True
),
Element(
	name=Valanginian,
	color=#008000,
	is_active=True
),
Element(
	name=Kimmeridgian,
	color=#00FF00,
	is_active=True
)
]
)
],
	fault_relations=
[[False]],

In [4]:
#Plots the model in 3D in XZ plane, replace plot_3d() by plot_2D() for XZ plot
#The plot is not restricted like the model is ! Points WILL appear outside of defined extent with high angle value (>40°)
fig = gpv.plot_3d(
    monoclinal_model,
    show_data_points=True,
    show_orientations=True,
    show_title=True,
    show_legend=True
)

In [5]:
#Computes model
monoclinal_model.update_transform(gp.data.GlobalAnisotropy.NONE)
gp.compute_model(monoclinal_model, engine_config=gp.data.GemPyEngineConfig())

#Basic 3D plot with all the information, execute celle bellow for a less busy plot
gpv.plot_3d(monoclinal_model, cell_number='mid')


Setting Backend To: AvailableBackends.PYTORCH
Chunking done: 73 chunks
Chunking done: 15 chunks


C:\Users\mael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\gempy_viewer\modules\plot_3d\drawer_surfaces_3d.py:38: PyVistaDeprecationWarning: 
../../AppData/Local/Packages/PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0/LocalCache/local-packages/Python313/site-packages/gempy_viewer/modules/plot_3d/drawer_surfaces_3d.py:38: Argument 'color' must be passed as a keyword argument to function 'BasePlotter.add_mesh'.
From version 0.50, passing this as a positional argument will result in a TypeError.
  gempy_vista.surface_actors[element.name] = gempy_vista.p.add_mesh(


In [15]:
gpv.plot_3d(monoclinal_model, cell_number='mid', show_data= False, show_lith = False) #Only see the layers